# Predicting NBA MVP Vote Share and Ranking Candidates

This notebook is the guided, reproducible entry point for the final CRISP-DM workflow. It delegates the full audited implementation to the version-controlled programs in `src/`, then loads their machine-readable results for interpretation. This avoids maintaining a second, silently different modeling implementation inside the notebook.

**Research question:** Can regular-season player statistics and team performance predict continuous `award_share` and correctly rank MVP candidates within each season?

## 1. Environment and data contract

Download the Kaggle ZIP as described in `DATA_SOURCE.md`. The default location is `data/raw/nba_mvp_stats.zip`; the `NBA_MVP_DATA_PATH` environment variable can override it.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import importlib.metadata as metadata
import os
import subprocess
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_PATH = Path(os.environ.get('NBA_MVP_DATA_PATH', REPO_ROOT / 'data/raw/nba_mvp_stats.zip')).resolve()
RESULTS_DIR = REPO_ROOT / 'results'
MODEL_PATH = REPO_ROOT / 'models/nba_mvp_production_models_through_2022.joblib'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}. Follow DATA_SOURCE.md first.')

print('Repository:', REPO_ROOT)
print('Dataset:', DATA_PATH)
for package in ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'joblib']:
    print(f'{package}: {metadata.version(package)}')

## 2. Data understanding

The raw table is inspected before preparation. The sparse target means all-player error alone is insufficient; later sections add vote-recipient and within-season ranking metrics.

In [ ]:
with ZipFile(DATA_PATH) as archive:
    members = [name for name in archive.namelist() if name.lower().endswith('.csv')]
    if len(members) != 1:
        raise ValueError(f'Expected exactly one CSV, found: {members}')
    with archive.open(members[0]) as stream:
        raw = pd.read_csv(stream)

overview = pd.Series({
    'rows': len(raw),
    'columns': raw.shape[1],
    'first_season': int(raw['season'].min()),
    'last_season': int(raw['season'].max()),
    'seasons': int(raw['season'].nunique()),
    'positive_vote_rows': int(raw['award_share'].gt(0).sum()),
    'zero_target_pct': 100 * raw['award_share'].eq(0).mean(),
    'tot_rows': int(raw['team_id'].eq('TOT').sum()),
    'exact_duplicate_rows': int(raw.duplicated().sum()),
})
print(overview.to_string())

## 3. Reproduce the locked chronological evaluation

The final program rebuilds the 100- and 500-minute cohorts, performs leakage-safe era feature engineering, refits only on 1982–2018, and evaluates once on 2019–2022. It writes all comparison, ranking, sensitivity, and guardrail tables to `results/`.

In [ ]:
env = os.environ.copy()
env.update({
    'NBA_MVP_DATA_PATH': str(DATA_PATH),
    'NBA_MVP_RESULTS_DIR': str(RESULTS_DIR),
    'NBA_MVP_DEPLOYMENT_DIR': str(RESULTS_DIR / 'deployment'),
    'NBA_MVP_MODEL_PATH': str(MODEL_PATH),
})
completed = subprocess.run(
    [sys.executable, str(REPO_ROOT / 'src/locked_test_evaluation.py')],
    cwd=REPO_ROOT, env=env, text=True, capture_output=True, check=True,
)
print('Locked evaluation: PASS')
print('\n'.join(completed.stdout.splitlines()[-12:]))

## 4. Compare vote-share prediction and season-level ranking

The champion is selected for the combined objective, not one isolated score. Lower RMSE is better; higher NDCG@5 is better; a winner rank of 1 is ideal.

In [ ]:
comparison = pd.read_csv(RESULTS_DIR / 'locked_test_model_comparison.csv')
reported = comparison.loc[comparison['model'].isin([
    'WS OLS baseline', 'Weighted Ridge', 'Histogram GB', 'Extra Trees hurdle'
]), [
    'model', 'rmse_all', 'rmse_positive', 'r2_all', 'mean_ndcg_at_5',
    'mean_winner_rank', 'fractional_top1_accuracy', 'season_total_mae',
    'mean_winner_share_bias', 'final_clipping_pct'
]]
print(reported.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, name in zip(axes, ['locked_test_regression.png', 'locked_test_ranking.png']):
    ax.imshow(plt.imread(RESULTS_DIR / name))
    ax.axis('off')
plt.tight_layout()
plt.show()

### Interpretation

Histogram GB is the official champion because it has the strongest vote-share accuracy, ties the best serious models on winner rank and top-1 accuracy, and underpredicts winners less than the hurdle challenger. Extra Trees hurdle remains a required challenger because it has the highest NDCG@5 and best season-total calibration. Weighted Ridge is retained for interpretation and ranking comparison, not production calibration.

In [ ]:
winner_audit = pd.read_csv(RESULTS_DIR / 'locked_test_winner_audit.csv')
winner_audit = winner_audit.loc[winner_audit['model'].isin([
    'WS OLS baseline', 'Weighted Ridge', 'Histogram GB', 'Extra Trees hurdle'
])]
print(winner_audit[['model', 'season', 'actual_winner', 'predicted_top',
                    'winner_rank', 'ndcg_at_5']].round(4).to_string(index=False))

## 5. Production refit and artifact checks

Only after the locked evaluation is complete, the frozen champion and challenger are refit through 2022. The deployment program performs schema-gate and target-free scoring tests before saving the artifact.

In [ ]:
completed = subprocess.run(
    [sys.executable, str(REPO_ROOT / 'src/deployment_readiness.py')],
    cwd=REPO_ROOT, env=env, text=True, capture_output=True, check=True,
)
artifact = joblib.load(MODEL_PATH)
artifact_features = artifact['numeric_features'] + artifact['categorical_features']
print('Deployment refit: PASS')
print('Artifact load: PASS')
print('Trained through:', artifact['trained_through'])
print('Target present among features:', 'award_share' in artifact_features)
print('Artifact bytes:', MODEL_PATH.stat().st_size)

## 6. Conclusion and limitations

Regular-season statistics and team context predict meaningful MVP signal, but the four-season locked test contains only two unique winners. The champion clips many small negative raw predictions to zero and underpredicts winner shares. Human voting also depends on narrative and eligibility information absent from the data. Treat the model as a statistical decision aid, preserve the challenger, and evaluate genuinely new seasons without retuning on them first.

See `MODEL_CARD.md` for the full risk and monitoring policy and `reports/NBA_MVP_CRISP_DM_Final_Report.pdf` for the research report.